# 01 — Exploración del dataset BCSC

**Objetivo:** cargar los 3 CSVs del *BCSC Risk Factor Dataset*, entender la estructura, decodificar variables y generar `data/clean/bcsc_clean.parquet` listo para EDA y modelado.

Fuente: https://www.bcsc-research.org/index.php/datasets/rf/risk-factor-dataset-download

In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path('../data/raw')
CLEAN = Path('../data/clean')
CLEAN.mkdir(parents=True, exist_ok=True)

files = sorted(RAW.glob('bcsc_risk_factors_summarized*.csv'))
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(df.shape)
df.head()

In [ ]:
df.info()
print('\nNulos por columna:')
df.isna().sum()

## Codificación BCSC

El dataset usa códigos numéricos. `9` siempre = desconocido. Resumen:

- **age_group_5_years**: 1=18-29, 2=30-34, 3=35-39, 4=40-44, 5=45-49, 6=50-54, 7=55-59, 8=60-64, 9=65-69, 10=70-74, 11=75-79, 12=80-84, 13=≥85
- **race_eth**: 1=Blanca no hispana, 2=Negra no hispana, 3=Asiática/PI, 4=Nativa, 5=Hispana, 6=Otra/mixta, 9=desconocida
- **first_degree_hx**: 0=no, 1=sí, 9=desconocido
- **age_menarche**: 0=≥14, 1=12-13, 2=<12, 9=desc.
- **age_first_birth**: 0=<20, 1=20-24, 2=25-29, 3=≥30, 4=Nulípara, 9=desc.
- **BIRADS_breast_density**: 1=casi todo graso, 2=densidad fibroglandular dispersa, 3=heterogéneamente densa, 4=extremadamente densa, 9=desc.
- **current_hrt**: 0=no, 1=sí, 9=desc.
- **menopaus**: 1=pre/peri, 2=post (natural), 3=post (quirúrgica), 9=desc.
- **bmi_group**: 1=10-24.99, 2=25-29.99, 3=30-34.99, 4=≥35, 9=desc.
- **biophx**: 0=no biopsia previa, 1=sí, 9=desc.
- **breast_cancer_history**: 0=sin diagnóstico, 1=diagnosticada (target)
- **count**: nº de mamografías agregadas con esta combinación de factores

In [ ]:
# Total de mamografías representadas
print('Filas (combinaciones únicas):', len(df))
print('Mamografías representadas:', int(df['count'].sum()))
print('\nDistribución target (ponderada por count):')
df.groupby('breast_cancer_history')['count'].sum()

In [ ]:
# Eliminar filas con target desconocido (no debería haber, pero por seguridad)
df = df[df['breast_cancer_history'].isin([0, 1])].copy()
df.to_parquet(CLEAN / 'bcsc_clean.parquet', index=False)
print('Guardado en data/clean/bcsc_clean.parquet')